# Portfolio SLM Fine-tuning
**Model:** Qwen2.5-1.5B-Instruct + QLoRA → GGUF

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Update `DRIVE_DATASET_PATH` in Cell 4
3. Update `HF_USERNAME` and `HF_TOKEN` in Cell 8

Then: Runtime → Run all. Takes ~1.5 hours.

In [ ]:
# Cell 1: Install dependencies (~3 minutes)
%%capture
!pip install unsloth
!pip install --upgrade trl transformers accelerate
print('Done.')

In [ ]:
# Cell 2: Load base model
# Downloads Qwen2.5-1.5B-Instruct (~3GB). Takes ~2 minutes.
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("Model loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 3: Attach LoRA adapters
# Adds small trainable layers on top of the frozen base model.
# Only ~3M parameters trained instead of 1.5B.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()
# Expected: trainable params ~3M | all params ~1.5B | trainable% ~0.2%

In [ ]:
# Cell 4: Load dataset from Google Drive
# UPDATE: change the path to where you uploaded qa_dataset.jsonl
from google.colab import drive
drive.mount("/content/drive")

from datasets import load_dataset

DRIVE_DATASET_PATH = "/content/drive/MyDrive/qa_dataset.jsonl"  # <-- UPDATE THIS

dataset = load_dataset("json", data_files=DRIVE_DATASET_PATH, split="train")
print(f"Dataset loaded: {len(dataset)} examples")
print("\nSample question:", dataset[0]["messages"][1]["content"])
print("Sample answer:  ", dataset[0]["messages"][2]["content"][:100], "...")

In [ ]:
# Cell 5: Format dataset for Qwen2.5 chat template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def format_record(record):
    return tokenizer.apply_chat_template(
        record["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

dataset = dataset.map(lambda r: {"text": format_record(r)}, batched=False)

print("Formatted. Sample:")
print(dataset[0]["text"][:300])

In [ ]:
# Cell 6: Train (~1 to 1.5 hours on T4 GPU)
# Watch the loss column — it should drop from ~2.0 down to ~0.3-0.5
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/tmp/portfolio-slm-checkpoints",
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"\nDone. Final loss: {stats.training_loss:.4f}")
print("Good if loss < 0.5. Rerun with more epochs if > 0.8.")

In [ ]:
# Cell 7: Quality check — ask 5 questions before uploading
# If answers look wrong, DO NOT proceed to Cell 8. Investigate first.
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are Abhinav Prakash (handle: psychopunksage). "
    "You are a systems engineer focused on blockchain infrastructure, "
    "low-level systems programming, and Linux kernel development. "
    "You write Rust, Go, Solidity, and C. You are direct, technically precise, "
    "with dry humor. You have no patience for unnecessary abstraction. "
    "Answer all questions about yourself in first person."
)

def ask(question):
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": question}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=200,
                         temperature=0.3, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

questions = [
    "Where do you currently work?",
    "What's your experience with Rust?",
    "Tell me about the relay pipeline you redesigned.",
    "What's your dev setup like?",
    "What kind of work are you looking for?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print()

In [ ]:
# Cell 8: Export to GGUF and upload to HuggingFace Hub
# Only run this if Cell 7 answers looked correct.
# UPDATE HF_USERNAME and HF_TOKEN before running.

HF_USERNAME = "your-hf-username"          # <-- UPDATE THIS
HF_TOKEN    = "hf_xxxxxxxxxxxxxxxxxxxx"   # <-- UPDATE THIS (HF Settings > Access Tokens)

REPO_ID       = f"{HF_USERNAME}/abhinav-portfolio-slm"
GGUF_FILENAME = "abhinav-portfolio.Q4_K_M.gguf"
GGUF_DIR      = "/tmp/portfolio-gguf"

print("Exporting to GGUF (Q4_K_M quantization)... ~16 minutes")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

# Unsloth appends _gguf to the directory — search both locations
import os, glob
gguf_files = []
for d in [GGUF_DIR, f"{GGUF_DIR}_gguf"]:
    if os.path.exists(d):
        gguf_files = glob.glob(f"{d}/*.gguf")
        if gguf_files:
            break

if not gguf_files:
    raise FileNotFoundError(f"No .gguf file found. Searched: {GGUF_DIR} and {GGUF_DIR}_gguf")

gguf_path = gguf_files[0]
print(f"Found GGUF: {gguf_path}")

print(f"\nUploading to https://huggingface.co/{REPO_ID} ...")
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=gguf_path,
    path_in_repo=GGUF_FILENAME,
    repo_id=REPO_ID,
)

print(f"\nDone. Model at: https://huggingface.co/{REPO_ID}")
print(f"File: {GGUF_FILENAME}")
print("Copy the repo ID and filename — you'll need both for Phase 3 (Modal).")